# RoboTwin supervisor pilot

Run the verified setup stages below. Upload both `robotwin-colab-input.zip` and `robotwin-pilot-code.zip` to session storage. Add `GEMINI_API_KEY` in Colab Secrets and enable notebook access. The development gate must pass before the 40-episode exploratory comparison. No model weights are trained. See `docs/robotwin-pilot.md`.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, os, subprocess, sys
bundle = Path('/content/robotwin-colab-input.zip')
if not bundle.is_file():
    uploaded = files.upload()  # Or upload using the Files sidebar before running this cell.
    assert 'robotwin-colab-input.zip' in uploaded
    bundle.write_bytes(uploaded['robotwin-colab-input.zip'])
root = Path('/content/robotwin-agent')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for member in z.infolist():
        assert (root / member.filename).resolve().is_relative_to(root.resolve())
    z.extractall(root)
os.chdir(root)
def run(*args):
    process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)
print(root)


## 1. Check the assigned runtime before large downloads
Installs only Vulkan diagnostics and basic system libraries. If the next cell fails,
retain evidence/preflight.json. Do not install a replacement NVIDIA kernel driver
inside Colab. A software renderer is not an acceptable pass.


In [ ]:
run('apt-get', 'update', '-qq')
run('apt-get', 'install', '-y', '-qq', 'vulkan-tools', 'libvulkan1', 'libgl1', 'libglib2.0-0', 'ffmpeg', 'git', 'unzip', 'build-essential')
import runpy
runpy.run_path('setup/colab_graphics.py', run_name='__main__')
run(sys.executable, 'setup/preflight.py')


## 2. Check compiler prerequisites
The client specifies CUDA 12.x with development headers and GCC/G++ 13 or newer.
If the runtime lacks these, stop and resolve the toolchain before installing CuRobo.
Do not assume the CUDA version printed by nvidia-smi is an installed toolkit.


In [ ]:
import shutil, re
run('apt-get', 'install', '-y', '-qq', 'cuda-nvcc-12-8', 'cuda-cudart-dev-12-8', 'cuda-cccl-12-8')
os.environ['PATH'] = '/usr/local/cuda-12.8/bin:' + os.environ['PATH']
nvcc = shutil.which('nvcc')
assert nvcc, 'CUDA development toolkit is missing'
cuda_home = Path(nvcc).resolve().parent.parent
os.environ['CUDA_HOME'] = str(cuda_home)
assert (cuda_home / 'include/cuda_runtime_api.h').is_file(), 'CUDA headers missing'
nvcc_version = subprocess.check_output([nvcc, '--version'], text=True)
assert 'release 12.' in nvcc_version, nvcc_version
cxx = shutil.which('g++-13') or shutil.which('g++')
assert cxx
major = int(subprocess.check_output([cxx, '-dumpversion'], text=True).split('.')[0])
assert major >= 13, 'Client requires GCC/G++ >=13; this runtime needs toolchain work'
os.environ['CXX'] = cxx
os.environ['CC'] = cxx.replace('g++', 'gcc')
run(sys.executable, '-m', 'pip', 'install', 'uv')


## 3. Install isolated environments and test real SAPIEN rendering
The first render precedes the full dependency stack. A produced smoke image is only
a renderer check, not a RoboTwin task episode. Installation can take substantial time.


In [ ]:
run('bash', 'setup/install_linux.sh', 'environments')


In [ ]:
from IPython.display import display, Image
display(Image(filename='evidence/sapien-smoke.png'))


## 4. Fetch pinned source, compile CuRobo, and download assets
The client estimates the asset download at roughly 16 GB.


In [ ]:
run('bash', 'setup/install_linux.sh', 'source')
run('bash', 'setup/install_linux.sh', 'curobo')
run('bash', 'setup/install_linux.sh', 'assets')
run('bash', 'setup/install_linux.sh', 'weights')


## Verify the supervisor integration
Uses development seeds, not the evaluation seeds. Stops on any failed gate. The secret is never printed or exported.


In [ ]:
from google.colab import userdata
with zipfile.ZipFile('/content/robotwin-pilot-code.zip') as archive:
    for item in archive.infolist():
        assert (Path('/content') / item.filename).resolve().is_relative_to(Path('/content'))
    archive.extractall('/content')
os.environ['PYTHONPATH'] = '/content/src'
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
run('.venv-policy/bin/python', '-m', 'unittest', 'discover', '-s', '/content/pilot-tests')
run('.venv-policy/bin/python', '-u', 'run_development.py', '--output', 'outputs/development')


## Run the frozen comparison
Ten starts, four conditions, 40 episodes. An invalid trial stops the batch and remains in the results. Do not overwrite output directories.


In [ ]:
run('.venv-policy/bin/python', '-u', 'run_pilot.py', '--readiness', 'outputs/development/readiness.json', '--output', 'outputs/pilot')
run('.venv-policy/bin/python', 'setup/verify_pilot_results.py', 'outputs/pilot')


## Export before releasing the runtime
The archive contains only the two experiment output folders and runtime evidence; it excludes credentials.


In [ ]:
archive_path = Path('/content/robotwin-pilot-results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path('outputs/development'), Path('outputs/pilot'), Path('evidence')):
        for path in sorted(folder.rglob('*')):
            if path.is_file():
                archive.write(path, path.as_posix())
print('Evidence archive bytes:', archive_path.stat().st_size)
files.download(str(archive_path))
